In [ ]:
import numpy as np
import cv2
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.utils import flow_to_image
import torch
import torch.nn.functional as Fun

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Farneback in Numpy

In [ ]:
def OF_Ferneback_np(frames, pyr=0.5, lv=3, win=10, it=10, poly=5, sig=1.2):
    flows = []
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    
    for i in range(len(frames) - 1):
        next_gray = cv2.cvtColor(frames[i+1], cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, next_gray, None, pyr, lv, win, it, poly, sig, 0
        )
        flows.append(flow)
        prev_gray = next_gray
    return flows

# RAFT Model in Tensor

In [ ]:
# Prepare images (Must be RGB, and dimensions divisible by 8)
# img1, img2 should be tensors of shape (1, 3, H, W)
# Standardize images to [-1, 1] as required by RAFT
def OF_RAFT_tensor(img1,img2):
    weights = Raft_Large_Weights.DEFAULT
    model = raft_large(weights=weights, progress=False).to(device).eval()
    with torch.no_grad():
        list_of_flows = model(2.0*img1.to(dtype=torch.float)-1.0, 2.0*img2.to(dtype=torch.float)-1.0)
        predicted_flow = list_of_flows[-1]
    return predicted_flow

# Optical Flow Field to Polar in Numpy (Magnitude & Angle)

In [ ]:
def OF2Polar_np(flows, Deg=True):
    mags, angs = [], []
    for flow in flows:
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=Deg)
        mags.append(mag)
        angs.append(ang)
    return mags, angs

# Optical Flow FIeld Visuzlization in HSV as Numpy

In [ ]:
def OF_visual_HSV_np(flows, Deg=True):
    rgb_list = []
    for flow in flows:
        # Initialize HSV canvas
        hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=Deg)
        hsv[..., 0] = ang / 2 if Deg else ang * 180 / np.pi / 2
        hsv[..., 1] = 255
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        rgb_list.append(cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR))
    return rgb_list

# Draw Optical Flow Vector Field in Numpy

In [ ]:
def draw_optical_flow_np(flow_list, frame_list, step=10, color=(0, 255, 0), line_thick=1, point_thick=1):
    output_frames = []
    # Iterate through paired flow fields and frames
    for flow, frame in zip(flow_list, frame_list):
        # Create a copy to avoid modifying the original list elements in-place
        img = frame.copy()
        h, w = img.shape[:2]
        # Create the grid of points
        y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2, -1).astype(int)
        fx, fy = flow[y, x].T
        # Create a mask for the lines
        mask = np.zeros_like(img)
        # Calculate endpoints
        lines = np.vstack([x, y, x + fx, y + fy]).T.reshape(-1, 2, 2)
        lines = np.int32(lines + 0.5)
        # Draw the vectors
        for (x1, y1), (x2, y2) in lines:
            cv2.line(mask, (x1, y1), (x2, y2), color, line_thick)
            cv2.circle(img, (x2, y2), point_thick, color, -1)
        # Combine the mask and the frame
        drawn_frame = cv2.add(img, mask)
        output_frames.append(drawn_frame) 
    return output_frames

# Image Warping for Numpy

In [ ]:
def warp_OF_np(image, flow):
    h, w = flow.shape[:2]
    y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    new_x = (x + flow[..., 0]).astype(np.float32)
    new_y = (y + flow[..., 1]).astype(np.float32)
    warped_image = cv2.remap(image, new_x, new_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    return warped_image

# Image Warping for Tensor

In [ ]:
def warp_OF_tensor(image, flow):
    if image.dim() == 3:
        image = image.unsqueeze(0)
    if image.dtype == torch.uint8:
        image = image.float() / 255.0
    B, C, H, W = image.shape
    device = image.device
    flow = flow.to(device)
    grid_y, grid_x = torch.meshgrid(torch.arange(H, device=device, dtype=torch.float32),torch.arange(W, device=device, dtype=torch.float32),indexing='ij')
    grid = torch.stack((grid_x, grid_y), dim=-1)
    grid = grid.unsqueeze(0).expand(B, -1, -1, -1)
    flow_permuted = flow.permute(0, 2, 3, 1)
    v_grid = grid + flow_permuted
    scale = torch.tensor([W - 1, H - 1], device=device, dtype=torch.float32)
    v_grid_norm = 2.0 * v_grid / scale - 1.0
    warped_image = Fun.grid_sample(image,v_grid_norm,mode='bilinear',padding_mode='zeros',align_corners=True)
    return warped_image

In [ ]:
def get_occlusion_errors(frames, farneback_params=None):
    if farneback_params is None:
        farneback_params = {}
    if len(frames) < 2:
        return []
    grays = []
    for f in frames:
        if f.ndim == 3:
            gray = cv2.cvtColor(f.astype(np.uint8), cv2.COLOR_BGR2GRAY)
        else:
            gray = f.astype(np.uint8)
        grays.append(gray)
    H, W = grays[0].shape
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    fb_errors = []
    for t in range(len(frames) - 1):
        g0, g1 = grays[t], grays[t + 1]
        # forward and backward flows
        flow_fwd = cv2.calcOpticalFlowFarneback(g0, g1, None, 0.5, 3, 15, 3, 5, 1.2, 0, **farneback_params)
        flow_bwd = cv2.calcOpticalFlowFarneback(g1, g0, None, 0.5, 3, 15, 3, 5, 1.2, 0, **farneback_params)
        # forward-warp pixel locations
        map_x = (grid_x + flow_fwd[..., 0]).astype(np.float32)
        map_y = (grid_y + flow_fwd[..., 1]).astype(np.float32)
        # sample backward flow at forward-warped coords
        bwd_x = cv2.remap(flow_bwd[..., 0], map_x, map_y,interpolation=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        bwd_y = cv2.remap(flow_bwd[..., 1], map_x, map_y,interpolation=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        sampled_bwd = np.stack([bwd_x, bwd_y], axis=-1)
        # forward-backward error (float, not thresholded)
        fb_error = np.linalg.norm(flow_fwd + sampled_bwd, axis=2)
        fb_errors.append(fb_error.astype(np.float32))
    return fb_errors